# ARIMA Selection and Diagnostic Workflow

Module: Quantitative Methods and Financial Time Series

## Lesson summary

This lab turns the pre-model evidence from Lesson 2.3 into a declared ARIMA
selection rule. Candidate orders are ranked on an earlier training sample,
the selected mean model is compared with a chronological benchmark on a later
holdout, and a full-sample refit is checked for residual autocorrelation and
non-Gaussianity {cite}`box2015time,hamilton1994time`.

## Learning objectives

By the end of this lab, readers should be able to:

- distinguish the order-selection sample from the final holdout;
- select the lowest-AIC admissible candidate in a declared grid and report
  whether BIC agrees;
- verify convergence, stationarity, and invertibility before interpretation;
- compare a fixed-origin ARIMA holdout forecast with a training-mean benchmark;
- run residual autocorrelation and normality checks; and
- write a model-selection conclusion that separates in-sample fit,
  out-of-sample error, and remaining model risk.

## Prerequisites

Lesson 2.3 establishes the stationarity decision and explains why its
Ljung-Box calculation on raw returns is a pre-model screen. This lab assumes
readers can interpret ADF, ACF, and PACF without treating a p-value as proof
of a model.

## Model and decision rule

An ARIMA model applies autoregressive and moving-average structure after
differencing:

$$
\phi(L)(1-L)^d y_t = c + \theta(L)\varepsilon_t.
$$

The USD/MXN FIX return series is modeled with $d=0$. The grid below admits a
candidate only when optimization converges and its AR and MA representations
are stationary and invertible. The predeclared rule selects the lowest-AIC
admissible candidate on the first 80% of observations; BIC and holdout MAE are
reported as separate evidence, not silently folded into the choice.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import mean_absolute_error
from statsmodels.tsa.arima.model import ARIMA

from src.market_data import banxico_daily_panel
from src.market_data_quality import log_returns
from src.module2_visuals import build_residual_diagnostics
from src.time_series_diagnostics import (
    adf_report,
    arima_order_search,
    jarque_bera_report,
    ljung_box_report,
)

REQUESTED_START = "2021-01-01"
REQUESTED_END = "2026-06-05"
SIGNIFICANCE_LEVEL = 0.05
TRAINING_FRACTION = 0.80

## Provider-dated Banxico FIX observations

The committed snapshot preserves Banxico publication dates. Missing calendar
dates are not forward-filled before returns are calculated, so a zero return
is never created merely to align this series with another panel
{cite}`banxicoSIE2025`.

In [ ]:
banxico = banxico_daily_panel(start=REQUESTED_START, end=REQUESTED_END)
prices = banxico["usd_mxn"].dropna().rename("usd_mxn_fix_mxn_per_usd")
returns = log_returns(prices).rename("usd_mxn_fix_log_return")
calendar_gaps = prices.index.to_series().diff().dt.days.dropna()

sample_inventory = pd.DataFrame(
    [
        {
            "provider_series": "Banxico SIE SF43718",
            "data_mode": banxico.attrs["data_mode"],
            "snapshot_generated_at": banxico.attrs["snapshot_generated_at"],
            "observed_start": prices.index.min().date().isoformat(),
            "observed_end": prices.index.max().date().isoformat(),
            "level_observations": len(prices),
            "return_observations": len(returns),
            "maximum_calendar_gap_days": int(calendar_gaps.max()),
            "return_unit": "log change per FIX publication interval",
            "observation_policy": banxico.attrs["observation_policy"],
        }
    ]
)
sample_inventory

**Output interpretation.**

The inventory reports actual inclusive sample boundaries and the longest gap
between consecutive FIX publications. Returns span publication intervals,
which are usually one business day but can cross weekends or holidays. They
are not returns from a mechanically forward-filled business-day panel.

In [ ]:
pd.DataFrame({"level_mxn_per_usd": prices, "log_return": returns}).head()

**Output interpretation.**

The first valid log return requires two consecutive published FIX levels.
Banxico FIX is an official exchange-rate fixing, not an adjusted equity close
or a guaranteed executable transaction price.

## Stationarity decision

In [ ]:
level_adf = adf_report(prices)
return_adf = adf_report(returns)
adf_decision = pd.DataFrame(
    {
        "series": ["FIX level", "FIX log return"],
        "adf_statistic": [level_adf["statistic"], return_adf["statistic"]],
        "p_value": [level_adf["p_value"], return_adf["p_value"]],
        "decision_at_5pct": [
            "reject unit-root null"
            if report["p_value"] < SIGNIFICANCE_LEVEL
            else "do not reject unit-root null"
            for report in (level_adf, return_adf)
        ],
        "used_lag": [int(level_adf["used_lag"]), int(return_adf["used_lag"])],
        "observations": [int(level_adf["nobs"]), int(return_adf["nobs"])],
    }
).set_index("series")
adf_decision

**Output interpretation.**

The 5% decision column states the evidence without converting failure to
reject into proof. The level and return tests use a constant deterministic
term; a different trend specification is a sensitivity check, not a hidden
way to force the desired conclusion.

## Chronological training and holdout samples

In [ ]:
split_index = int(len(returns) * TRAINING_FRACTION)
training_returns = returns.iloc[:split_index].copy()
testing_returns = returns.iloc[split_index:].copy()
assert training_returns.index.max() < testing_returns.index.min()

pd.DataFrame(
    {
        "sample": ["training", "holdout"],
        "start": [training_returns.index.min(), testing_returns.index.min()],
        "end": [training_returns.index.max(), testing_returns.index.max()],
        "observations": [len(training_returns), len(testing_returns)],
    }
).set_index("sample")

**Output interpretation.**

Every holdout observation occurs after the order-selection sample. The
holdout remains unused until the candidate grid and lowest-AIC rule have been
fixed.

## Admissible ARIMA order search

The grid is intentionally small for classroom use. AIC and BIC estimate
different complexity penalties; neither is a forecast score
{cite}`akaike1974new,schwarz1978estimating`.

In [ ]:
candidate_results = arima_order_search(
    training_returns,
    p_values=range(0, 4),
    d_values=[0],
    q_values=range(0, 4),
)
admissible = candidate_results.loc[
    candidate_results["converged"]
    & candidate_results["stationary"]
    & candidate_results["invertible"]
].dropna(subset=["aic", "bic"])
if admissible.empty:
    raise RuntimeError("No converged, stationary, invertible ARIMA candidate was found")

aic_best = admissible.sort_values("aic").iloc[0]
bic_best = admissible.sort_values("bic").iloc[0]
selected_order = (int(aic_best["p"]), int(aic_best["d"]), int(aic_best["q"]))
bic_order = (int(bic_best["p"]), int(bic_best["d"]), int(bic_best["q"]))

candidate_results.head(10)

In [ ]:
pd.DataFrame(
    {
        "criterion": ["AIC selection rule", "BIC comparison"],
        "order": [selected_order, bic_order],
        "criterion_value": [aic_best["aic"], bic_best["bic"]],
        "same_order_as_aic": [True, bic_order == selected_order],
    }
).set_index("criterion")

**Output interpretation.**

“Selected” means lowest AIC inside this declared grid and training sample. The
BIC row makes agreement or disagreement visible. Neither row establishes that
the data-generating process is literally ARIMA or stable across regimes.

## Chronological fixed-origin holdout comparison

In [ ]:
training_model = ARIMA(
    training_returns.to_numpy(),
    order=selected_order,
    enforce_stationarity=True,
    enforce_invertibility=True,
).fit()
if not bool(training_model.mle_retvals.get("converged", True)):
    raise RuntimeError(f"Selected ARIMA{selected_order} did not converge on training data")

model_forecast = pd.Series(
    training_model.forecast(steps=len(testing_returns)),
    index=testing_returns.index,
    name="arima_forecast",
)
training_mean_forecast = pd.Series(
    training_returns.mean(),
    index=testing_returns.index,
    name="training_mean_forecast",
)

forecast_evidence = pd.DataFrame(
    {
        "model": [f"ARIMA{selected_order}", "training-mean benchmark"],
        "holdout_mae": [
            mean_absolute_error(testing_returns, model_forecast),
            mean_absolute_error(testing_returns, training_mean_forecast),
        ],
        "unit": [
            "log return per FIX publication interval",
            "log return per FIX publication interval",
        ],
        "holdout_start": [testing_returns.index.min()] * 2,
        "holdout_end": [testing_returns.index.max()] * 2,
        "holdout_observations": [len(testing_returns)] * 2,
    }
).set_index("model")
forecast_evidence

**Output interpretation.**

Holdout MAE asks whether the selected linear mean model improves on a simple
constant forecast over later dates. Both paths are produced once at the end of
training; they are fixed-origin multi-step forecasts, not a rolling refit that
consumes realized holdout returns. A small difference is not evidence of an
economically useful strategy: returns are noisy, no trading costs are modeled,
and this is only one historical split.

## Full-sample refit and residual diagnostics

After the order and holdout comparison are recorded, the chosen specification
is refitted to all available returns for descriptive residual diagnostics.

In [ ]:
full_model = ARIMA(
    returns.to_numpy(),
    order=selected_order,
    enforce_stationarity=True,
    enforce_invertibility=True,
).fit()
if not bool(full_model.mle_retvals.get("converged", True)):
    raise RuntimeError(f"Selected ARIMA{selected_order} did not converge on full data")

residuals = pd.Series(full_model.resid, index=returns.index, name="arima_residual")
parameter_table = pd.DataFrame(
    {
        "estimate": full_model.params,
        "standard_error": full_model.bse,
    },
    index=full_model.param_names,
).rename_axis("parameter")
parameter_table

**Output interpretation.**

The compact table reports only estimated parameters and uncertainty, avoiding
execution timestamps and wide diagnostic blocks that do not fit the book.
Coefficients describe conditional-mean dependence, not profitability or
causality.

In [ ]:
residual_ljung_box = ljung_box_report(
    residuals,
    lags=[5, 10, 20],
    model_df=selected_order[0] + selected_order[2],
)
residual_jarque_bera = jarque_bera_report(residuals)

residual_evidence = pd.DataFrame(
    {
        "diagnostic": [
            "Ljung-Box lag 5",
            "Ljung-Box lag 10",
            "Ljung-Box lag 20",
            "Jarque-Bera normality",
        ],
        "p_value": [
            residual_ljung_box.loc[5, "lb_pvalue"],
            residual_ljung_box.loc[10, "lb_pvalue"],
            residual_ljung_box.loc[20, "lb_pvalue"],
            residual_jarque_bera["p_value"],
        ],
        "decision_at_5pct": [
            *[
                "remaining linear autocorrelation"
                if residual_ljung_box.loc[lag, "lb_pvalue"] < SIGNIFICANCE_LEVEL
                else "no rejection of zero residual autocorrelations"
                for lag in (5, 10, 20)
            ],
            "reject Gaussian residuals"
            if residual_jarque_bera["p_value"] < SIGNIFICANCE_LEVEL
            else "do not reject Gaussian residuals",
        ],
    }
).set_index("diagnostic")
residual_evidence

In [ ]:
residual_figure = build_residual_diagnostics(
    residuals,
    title=f"ARIMA{selected_order} residual diagnostics",
    lags=20,
    source="Banxico SIE SF43718 snapshot",
    data_mode="provider-dated snapshot",
)
display(residual_figure)
plt.close(residual_figure)

**Output interpretation.**

The Ljung-Box rows evaluate remaining *linear* autocorrelation after adjusting
degrees of freedom for the selected AR and MA terms. The squared-residual ACF
in the figure can still reveal variance dependence even when the mean
diagnostics are satisfactory. Jarque-Bera rejection warns against Gaussian
interval or tail claims; it does not by itself invalidate the conditional-mean
specification.

## Model limitations

- The FIX series is an official reference fixing, not a continuously traded
  executable price.
- Publication intervals are not equally spaced in calendar time; weekend and
  holiday moves are contained in the next published change.
- ADF, information criteria, and residual tests are sample- and
  specification-dependent.
- The grid excludes seasonal, exogenous, nonlinear, and regime-switching mean
  models.
- One chronological holdout is more credible than in-sample fit but does not
  establish stable forecast value.
- ARIMA models can miss volatility clustering and heavy-tailed innovations.

## Checkpoint exercise

1. Report the AIC-selected order, the BIC-selected order, and whether they
   agree.
2. Compare the selected model's holdout MAE with the training-mean benchmark.
3. Fit the next-lowest-AIC admissible candidate on the same training sample,
   report its holdout MAE and 10-lag residual Ljung-Box p-value, and choose
   between the two without using the holdout to retune the grid.
4. State what Jarque-Bera implies for Gaussian intervals and name one risk it
   does not measure.

In [ ]:
runner_up = admissible.sort_values("aic").iloc[1]
runner_order = (
    int(runner_up["p"]),
    int(runner_up["d"]),
    int(runner_up["q"]),
)
runner_model = ARIMA(
    training_returns.to_numpy(),
    order=runner_order,
    enforce_stationarity=True,
    enforce_invertibility=True,
).fit()
runner_forecast = runner_model.forecast(steps=len(testing_returns))
runner_residuals = pd.Series(runner_model.resid, index=training_returns.index)

checkpoint_comparison = pd.DataFrame(
    {
        "order": [selected_order, runner_order],
        "training_aic": [aic_best["aic"], runner_up["aic"]],
        "holdout_mae": [
            forecast_evidence.loc[f"ARIMA{selected_order}", "holdout_mae"],
            mean_absolute_error(testing_returns, runner_forecast),
        ],
        "training_residual_ljung_box_p10": [
            ljung_box_report(
                pd.Series(training_model.resid, index=training_returns.index),
                lags=[10],
                model_df=selected_order[0] + selected_order[2],
            ).loc[10, "lb_pvalue"],
            ljung_box_report(
                runner_residuals,
                lags=[10],
                model_df=runner_order[0] + runner_order[2],
            ).loc[10, "lb_pvalue"],
        ],
    },
    index=["AIC-selected", "AIC runner-up"],
)
checkpoint_comparison

```{dropdown} Suggested answer and decision rule
Use the displayed AIC/BIC orders, MAEs, and residual p-values rather than
copying fixed numbers. Prefer the predeclared AIC model unless the evidence
reveals a failure that was part of the original admissibility rule. Holdout
performance can evaluate that rule but must not silently redefine it after
the outcomes are visible. Jarque-Bera addresses Gaussian shape, not serial
independence, volatility stability, liquidity, or economic value.
```

## Handoff

The next comparison fits ARCH and GARCH with a constant mean. That is aligned
with this page only when the selected mean specification is ARIMA(0,0,0). If a
refreshed snapshot selects nontrivial AR or MA terms, the ARCH/GARCH page
remains a didactic variance comparison; an applied workflow should fit the
selected mean and variance equations jointly rather than pretending raw
returns are already mean-model residuals.